<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week4_Hierarchical_Indexing_Groupby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

### Week 4: Getting Start with Pandas
---

**Question:**
- What capabilities does Python provide to make data cleaning and analysis fast and easy? 

**Objectives:**
- Distinguish <span style="color:red">pandas Series and DataFrame</span> data structures.
- Apply the essential functionality of DataFrame including <span style="color:red">indexing, selection, filtering</span>.
- <span style="color:red">Fill or drop</span> missing values in DataFrame.
- <span style="color:red">Apply functions</span> to DataFrame values by using map.
- Summarize and compute <span style="color:red">descriptive statistics</span>.

In [ ]:
# import libraries


## Hierarchical Indexing

Hierarchical indexing is an important feature of pandas that enables you to have multiple
(two or more) index levels on an axis. Somewhat abstractly, it provides a way for
you to work with higher dimensional data in a lower dimensional form. Let’s start
with a simple example; create a Series with a list of lists (or arrays) as the index.

```
data = pd.Series(np.random.randn(9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data
```

What you’re seeing is a prettified view of a Series with a MultiIndex as its index. The
“gaps” in the index display mean “use the label directly above”:

```
data.index
```

With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data:

```
data['b']
data['b':'c']
data.loc[['b', 'd']]
```

Selection is even possible from an “inner” level:

```
data.loc[:, 2]
```

Hierarchical indexing plays an important role in reshaping data and group-based
operations like forming a pivot table. For example, you could rearrange the data into
a DataFrame using its unstack method:

```
data.unstack()
```

The inverse operation of unstack is stack:

```
data.unstack().stack()
```

With a DataFrame, either axis can have a hierarchical index:

```
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
frame
```

The hierarchical levels can have names (as strings or any Python objects). If so, these
will show up in the console output:

```
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame
```

With partial column indexing you can similarly select groups of columns:

```
frame['Ohio']
```

A MultiIndex can be created by itself and then reused; the columns in the preceding
DataFrame with level names could be created like this:

```
MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                       names=['state', 'color'])
```

### Reordering and Sorting Levels
At times you will need to rearrange the order of the levels on an axis or sort the data
by the values in one specific level. The swaplevel takes two level numbers or names
and returns a new object with the levels interchanged (but the data is otherwise
unaltered):

```
frame.swaplevel('key1', 'key2')
```

sort_index, on the other hand, sorts the data using only the values in a single level.
When swapping levels, it’s not uncommon to also use sort_index so that the result is
lexicographically sorted by the indicated level:

```
frame.sort_index(level=1)
frame.swaplevel(0, 1).sort_index(level=0)
```

### Summary Statistics by Level
Many descriptive and summary statistics on DataFrame and Series have a level
option in which you can specify the level you want to aggregate by on a particular
axis. Consider the above DataFrame; we can aggregate by level on either the rows or
columns like so:

```
frame.sum(level='key2')
frame.sum(level='color', axis=1)
```

### Indexing with a DataFrame's columns
It’s not unusual to want to use one or more columns from a DataFrame as the row
index; alternatively, you may wish to move the row index into the DataFrame’s columns.
Here’s an example DataFrame:


```
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})
frame
````

DataFrame’s set_index function will create a new DataFrame using one or more of
its columns as the index:

```
frame2 = frame.set_index(['c', 'd'])
frame2
```

By default the columns are removed from the DataFrame, though you can leave them
in:

```
frame.set_index(['c', 'd'], drop=False)
```

reset_index, on the other hand, does the opposite of set_index; the hierarchical
index levels are moved into the columns:

```
frame2.reset_index()
```

## GroupBy Mechanics

Categorizing a dataset and applying a function to each group, whether an aggregation
or transformation, is often a critical component of a data analysis workflow. After
loading, merging, and preparing a dataset, you may need to compute group statistics
or possibly pivot tables for reporting or visualization purposes. pandas provides a
flexible groupby interface, enabling you to slice, dice, and summarize datasets in a
natural way.

![](https://i.imgur.com/ySkeo12.png)


Each grouping key can take many forms, and the keys do not have to be all of the
same type:
* A list or array of values that is the same length as the axis being grouped
* A value indicating a column name in a DataFrame
* A dict or Series giving a correspondence between the values on the axis being grouped and the group names
* A function to be invoked on the axis index or the individual labels in the index

Note that the latter three methods are shortcuts for producing an array of values to be
used to split up the object.

```
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})
df
```

Suppose you wanted to compute the mean of the data1 column using the labels from
key1. There are a number of ways to do this. One is to access data1 and call groupby
with the column (a Series) at key1:

```
grouped = df['data1'].groupby(df['key1'])
grouped
```

This grouped variable is now a GroupBy object. It has not actually computed anything
yet except for some intermediate data about the group key df['key1']. The idea is
that this object has all of the information needed to then apply some operation to
each of the groups. For example, to compute group means we can call the GroupBy’s
mean method:

```
grouped.mean()
```

If instead we had passed multiple arrays as a list, we’d get something different:

```
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means
```

Here we grouped the data using two keys, and the resulting Series now has a hierarchical
index consisting of the unique pairs of keys observed:

```
means.unstack().stack()
```

In this example, the group keys are all Series, though they could be any arrays of the
right length:

```
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([states, years]).mean()
```

Frequently the grouping information is found in the same DataFrame as the data you
want to work on. In that case, you can pass column names (whether those are strings,
numbers, or other Python objects) as the group keys:

```
df.groupby('key1').mean()
```

```
df.groupby(['key1', 'key2']).mean()
```

You may have noticed in the first case df.groupby('key1').mean() that there is no
key2 column in the result. Because df['key2'] is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, though it is possible to filter down to a subset, as
you’ll see soon.
Regardless of the objective in using groupby, a generally useful GroupBy method is
size, which returns a Series containing group sizes:

```
df.groupby(['key1', 'key2']).size()
```

### Iterating Over Groups
The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data. Consider the following:

```
for name, group in df.groupby('key1'):
    print(name)
    print(group)
```

In the case of multiple keys, the first element in the tuple will be a tuple of key values:

```
for (k1, k2), group in df.groupby(['key1', 'key2']):
    print((k1, k2))
    print(group)
```

Of course, you can choose to do whatever you want with the pieces of data. A recipe
you may find useful is computing a dict of the data pieces as a one-liner:

```
pieces = dict(list(df.groupby('key1')))
pieces['b']
```

By default groupby groups on axis=0, but you can group on any of the other axes.
For example, we could group the columns of our example df here by dtype like so:

```
df.dtypes
grouped = df.groupby(df.dtypes, axis=1)
```

We can print out the groups like so:

```
for dtype, group in grouped:
    print(dtype)
    print(group)
```

### Selecting a Column or Subset of Columns
Indexing a GroupBy object created from a DataFrame with a column name or array
of column names has the effect of column subsetting for aggregation. This means
that:

```
df.groupby('key1')['data1']
df.groupby('key1')[['data2']]
```
are syntactic suger for:

```
df['data1'].groupby(df['key1'])
df[['data2']].groupby(df['key1'])
```

Especially for large datasets, it may be desirable to aggregate only a few columns. For
example, in the preceding dataset, to compute means for just the data2 column and
get the result as a DataFrame, we could write:

```
df.groupby(['key1', 'key2'])[['data2']].mean()
```

The object returned by this indexing operation is a grouped DataFrame if a list or
array is passed or a grouped Series if only a single column name is passed as a scalar:

```
s_grouped = df.groupby(['key1', 'key2'])['data2']
s_grouped
s_grouped.mean()
```

### Grouping with Dicts and Series
Grouping information may exist in a form other than an array. Let’s consider another
example DataFrame:

```
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])
people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values
people
```

Now, suppose I have a group correspondence for the columns and want to sum
together the columns by group:

```
mapping = {'a': 'red', 'b': 'red', 'c': 'blue',
           'd': 'blue', 'e': 'red', 'f' : 'orange'}
```

Now, you could construct an array from this dict to pass to groupby, but instead we
can just pass the dict (I included the key 'f' to highlight that unused grouping keys
are OK):

```
by_column = people.groupby(mapping, axis=1)
by_column.sum()
```

The same functionality holds for Series, which can be viewed as a fixed-size mapping:

```
map_series = pd.Series(mapping)
map_series
people.groupby(map_series, axis=1).count()
```

### Grouping with Functions
Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. Any function passed as a group key will be called once per index
value, with the return values being used as the group names. More concretely, consider
the example DataFrame from the previous section, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the len
function:

```
people.groupby(len).sum()
```

Mixing functions with arrays, dicts, or Series is not a problem as everything gets converted
to arrays internally:

```
key_list = ['one', 'one', 'one', 'two', 'two']
people.groupby([len, key_list]).min()
```

### Grouping by Index Levels
A final convenience for hierarchically indexed datasets is the ability to aggregate
using one of the levels of an axis index. Let’s look at an example:

```
columns = pd.MultiIndex.from_arrays([['US', 'US', 'US', 'JP', 'JP'],
                                    [1, 3, 5, 1, 3]],
                                    names=['cty', 'tenor'])
hier_df = pd.DataFrame(np.random.randn(4, 5), columns=columns)
hier_df
```

To group by level, pass the level number or name using the level keyword:

```
hier_df.groupby(level='cty', axis=1).count()
```